In [1]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

df_raw = spark.read.option("header", True).option("inferSchema", True).csv(RAW_POPULATION)
print(df_raw.columns)
df_raw.show(5)

['Rank', 'CCA3', 'Country/Territory', 'Capital', 'Continent', '2022 Population', '2020 Population', '2015 Population', '2010 Population', '2000 Population', '1990 Population', '1980 Population', '1970 Population', 'Area (km²)', 'Density (per km²)', 'Growth Rate', 'World Population Percentage']
+----+----+-----------------+----------------+---------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------+-----------------+-----------+---------------------------+
|Rank|CCA3|Country/Territory|         Capital|Continent|2022 Population|2020 Population|2015 Population|2010 Population|2000 Population|1990 Population|1980 Population|1970 Population|Area (km²)|Density (per km²)|Growth Rate|World Population Percentage|
+----+----+-----------------+----------------+---------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------+--

In [3]:
df_clean = (
    df_raw
    .withColumnRenamed("CCA3", "iso_code")
    .withColumnRenamed("Country/Territory", "country")
    .withColumnRenamed("Continent", "continent")
    .withColumnRenamed("2020 Population", "population_2020")
    .withColumnRenamed("2022 Population", "population_2022")
    .withColumnRenamed("Growth Rate", "growth_rate")
    .select("iso_code", "country", "continent", "population_2020", "population_2022", "growth_rate")
)

print("Row count:", df_clean.count())
df_clean.show(5)

Row count: 234
+--------+--------------+---------+---------------+---------------+-----------+
|iso_code|       country|continent|population_2020|population_2022|growth_rate|
+--------+--------------+---------+---------------+---------------+-----------+
|     AFG|   Afghanistan|     Asia|       38972230|       41128771|     1.0257|
|     ALB|       Albania|   Europe|        2866849|        2842321|     0.9957|
|     DZA|       Algeria|   Africa|       43451666|       44903225|     1.0164|
|     ASM|American Samoa|  Oceania|          46189|          44273|     0.9831|
|     AND|       Andorra|   Europe|          77700|          79824|       1.01|
+--------+--------------+---------+---------------+---------------+-----------+
only showing top 5 rows



In [4]:
df_clean.write.mode("overwrite").parquet(SILVER_POPULATION)
print("Written to:", SILVER_POPULATION)

Written to: C:\covid_pipeline\silver\population
